# Number Detector Neural Network

The following code's purpose is to create a neural network that can detect what number is being written.

The steps to this neural network consists of 5 steps.

### 1. Set up the environment and the data.
We are using numpy and matplotlib as our libraries to help us create the neural netork. In addition, we need to download the database and preprocess it by flattening the data. The reason why we are flattening it is that we need it to be a 1D array so that the matrix math works out since we are multiplying the matrix by a weights matrix with 1 column. As a result, this is the only way for the math to work out.

### 2. Initialize W and b
W is our weight and b is our bias.

### 3. Define our math functions

## Environment setup and data import

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
import pandas as pd

mnist = fetch_openml('mnist_784')

## Initialize W and b

**W1, b1** map 784 inputs → `hidden_size` hidden neurons (ReLU). **W2, b2** map `hidden_size` → **10** output logits (digits 0–9). You can change `hidden_size` in training; the last layer stays 10 so it matches MNIST labels and one-hot encoding.

In [2]:
def init_params(hidden_size, num_classes=10):
    w1 = np.random.randn(hidden_size, 784) * 0.01
    b1 = np.zeros((hidden_size, 1))
    w2 = np.random.randn(num_classes, hidden_size) * 0.01
    b2 = np.zeros((num_classes, 1))

    return w1, b1, w2, b2

## Define our math functions

#### ReLU
ReLU is defined as max(0, Z). The reason for this is because we want to ensure that whatever Z output we get is positive. This solves the issue of negative Z's, since if we know that an output is bad and is negative, we don't want it affecting the other values that are created. This eliminates the noise of the negative numbers.

### Softmax
Softmax is an exponential function, so that means it will be louder if it is very wrong while quieter if it it's close. Additionally, it solves the issue of when the sum is 0 by making it e^Z.

### ReLU_deriv

In [3]:
def ReLU(Z):
    return np.maximum(0, Z)

def softmax(Z):
    exp = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp / np.sum(exp, axis = 0)

def ReLU_deriv(Z):
    return Z > 0


### Forward Prop
$Z_1$ = $w_1$x + $b_1$. This is the equation for computing the prediction for the first layer. It uses the result to compute the same thing for the second layer. We use softmax for $Z_2$ because we want to find the actual percentage it belongs to each neuron/number.

### One Hot
Creates a one hot encoding scheme to determine the values

### Backword Prop
Decide how much we need to update our values. We use the result of those to update our weights and biasas using the gradient descent formula. 

## Overview
The basic idea of everything here is utilizing the gradient descent formula so that we can head towards the direction of a minimized error.

In [4]:
def forward_prop(w1, b1, w2, b2, x):
    z1 = np.dot(w1, x) + b1
    a1 = ReLU(z1)

    z2 = np.dot(w2, a1) + b2
    a2 = softmax(z2)

    return z1, a1, z2, a2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, 10))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def backward_prop(z1, a1, z2, a2, w1, w2, X, Y):
    m = Y.size

    one_hot_Y = one_hot(Y)

    dZ2 = a2 - one_hot_Y
    # Weight gradient
    dW2 = 1 / m * dZ2.dot(a1.T)
    # Bias gradient
    db2 = 1 / m * np.sum(dZ2, axis = 1, keepdims = True)

    # Update neurons 
    dZ1 = w2.T.dot(dZ2) * ReLU_deriv(z1)

    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis = 1, keepdims = True)
    
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2



Preprocess

In [5]:
X_raw = mnist.data
Y_raw = mnist.target

if isinstance(Y_raw, pd.Series):
    Y = Y_raw.values
else:
    Y = np.array(Y_raw)

Y = Y.astype(np.int32).flatten()

if X_raw.shape[0] == 70000:
    X = X_raw.T
else:
    X = X_raw.T

X = np.array(X_raw).T/255

m = X.shape[1]
n_train = 60000
X_train, X_test = X[:, :n_train], X[:, n_train:]
Y_train, Y_test = Y[:n_train], Y[n_train:]




## Training
Use predefined functions and run certain amount of epochs on number of batch sizes.

In [ ]:
epochs = 20
batch_size = 64
alpha = 0.1
hidden_size = 5

w1, b1, w2, b2 = init_params(hidden_size)

def cross_entropy_loss(a2, Y):
    m = Y.size
    one_hot_Y = one_hot(Y)

    return -np.sum(one_hot_Y * np.log(a2 + 1e-8)) / m


n_train = X_train.shape[1]

for epoch in range(epochs):
    # Mini-batch loop
    for start in range(0, n_train, batch_size):
        end = start + batch_size
        X_batch = X_train[:, start:end]
        Y_batch = Y_train[start:end]

        z1, a1, z2, a2 = forward_prop(w1, b1, w2, b2, X_batch)
        dW1, db1, dW2, db2 = backward_prop(z1, a1, z2, a2, w1, w2, X_batch, Y_batch)
        w1, b1, w2, b2 = update_params(w1, b1, w2, b2, dW1, db1, dW2, db2, alpha)


    z1, a1, z2, a2 = forward_prop(w1, b1, w2, b2, X_train)
    loss = cross_entropy_loss(a2, Y_train)
    print(f"Epoch {epoch + 1}/{epochs}, loss: {loss:.4f}")

Epoch 1/20, loss: 0.3028
Epoch 2/20, loss: 0.2271
Epoch 3/20, loss: 0.1827
Epoch 4/20, loss: 0.1536
Epoch 5/20, loss: 0.1323
Epoch 6/20, loss: 0.1165
Epoch 7/20, loss: 0.1043
Epoch 8/20, loss: 0.0951
Epoch 9/20, loss: 0.0864
Epoch 10/20, loss: 0.0790
Epoch 11/20, loss: 0.0729
Epoch 12/20, loss: 0.0678
Epoch 13/20, loss: 0.0632
Epoch 14/20, loss: 0.0597
Epoch 15/20, loss: 0.0555
Epoch 16/20, loss: 0.0521
Epoch 17/20, loss: 0.0494
Epoch 18/20, loss: 0.0467
Epoch 19/20, loss: 0.0443
Epoch 20/20, loss: 0.0423


In [7]:
def get_predictions(a2):
    return np.argmax(a2, axis = 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / len(predictions)

z1, a1, z2, a2 = forward_prop(w1, b1, w2, b2, X_test)
predictions = get_predictions(a2)
accuracy = np.mean(predictions == Y_test)
print(f"Test accuracy: {accuracy * 100:.2f}%")

Test accuracy: 97.49%
